**Part 5 · Projects** · step 21 of 23  
[← 20 When to use what](../../04_When_To_Use_What/20_choosing_the_right_tool.md) · [🏠 Roadmap](../../README.md) · [22 Project: dynamic tool agent (part 1) →](../22_dynamic_tool_agent/01_db_schema_design.ipynb)

# 21 · Project — Natural-Language → SQL Inventory Agent

Ask questions in English; the agent writes SQL, runs it through a tool and returns structured rows.

```
"latest inventory item" ──► agent (gpt-4o) ──► query_db("SELECT ... ORDER BY created_at DESC LIMIT 1")
                                     ◄── rows ──┘
                         └──► SQLResult(output=[...])
```

**Concepts used:** `@tool`, `create_agent`, `system_prompt`, `response_format` (structured output).

**Data:** `inventory.db` (SQLite, 1 table `inventory`). Recreate it any time with
`python seed_inventory_db.py` from this folder.

In [ ]:
from dotenv import load_dotenv

# Reads API keys (OPENAI_API_KEY, GROQ_API_KEY, GOOGLE_API_KEY, TAVILY_API_KEY)
# from the .env file at the AgenticAI root. load_dotenv() walks up parent
# folders until it finds one, so every notebook can share a single .env.
load_dotenv()

In [ ]:
import sqlite3

DB_NAME = "inventory.db"

# Peek at the data the agent will query
with sqlite3.connect(DB_NAME) as conn:
    print(conn.execute("SELECT sql FROM sqlite_master WHERE name='inventory'").fetchone()[0])
    for row in conn.execute("SELECT * FROM inventory LIMIT 5"):
        print(row)

## The tool
The database is opened **read-only** (`mode=ro`) — the LLM writes the SQL, so never let it run
`DROP`/`DELETE` against real data. Errors are returned as text so the agent can fix its query and retry.

In [ ]:
from langchain.tools import tool


@tool
def query_db(query: str) -> list | str:
    """Run a read-only SQL query against the SQLite `inventory` table.

    Columns: id, product_name, category, sku, quantity, price, supplier, warehouse, created_at.
    """
    try:
        with sqlite3.connect(f"file:{DB_NAME}?mode=ro", uri=True) as conn:
            return conn.execute(query).fetchall()
    except Exception as error:
        return f"Query failed, fix the SQL and try again: {error}"

## The agent

In [ ]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain.messages import HumanMessage


class SQLResult(BaseModel):
    """Structured answer returned by the agent."""

    output: list = Field(description="List of all the rows that answer the question")


agent = create_agent(
    model="gpt-4o",
    system_prompt=(
        "You are a SQL expert. Turn the user's request into a SQLite query, "
        "run it with the query_db tool and return the results."
    ),
    tools=[query_db],
    response_format=SQLResult,
)

In [ ]:
resp = agent.invoke({"messages": [HumanMessage("latest inventory item")]})
resp["structured_response"].output

In [ ]:
# See which SQL the model wrote
for m in resp["messages"]:
    m.pretty_print()

In [ ]:
resp = agent.invoke({"messages": [HumanMessage("total stock value per warehouse, highest first")]})
resp["structured_response"].output

---
✅ **Done with 21 Project: SQL agent.** Next: [22 Project: dynamic tool agent (part 1) →](../22_dynamic_tool_agent/01_db_schema_design.ipynb)  
[🏠 Roadmap](../../README.md)